# Partie Ecology

Le projet consiste à retrouver le narratif caché dans les données. Chaque domaine détient un fragment. Pour le domaine Ecology, la question est :

> *Existe-t-il des sites où la population de chauves-souris présente des baisses anormales, et quelles en sont les causes (environnement, matériel, activités humaines) ?*

## Résumé des résultats

1. **Données** : 6 fichiers CSV concernant l'observation de chauves-souris, l'environnement, les sites de surveillance, et les rapports de terrain. Problèmes de qualité résolus : valeurs numériques stockées en texte, formats de dates mixtes, doublons.

2. **Détection initiale de baisses anormales** :
   - Méthode STL (Seasonal-Trend-Loess) avec période=12 pour extraire tendance et saisonnalité.
   - Calcul du z-score sur les résidus avec fenêtre mobile de 5 mois.
   - Plusieurs anomalies temporelles détectées mais approche peu concluante.

3. **Méthodes multivariées testées (abandonnées)** :
   - StandardScaler + IsolationForest sur 10 features.
   - LocalOutlierFactor (LOF) avec n_neighbors=20.
   - Ces méthodes ne réduisent pas suffisamment le nombre de sites et sont difficiles à interpréter.

4. **Méthode finale adoptée** :
   - Calcul du z-score sur les variations mensuelles de population.
   - 72 anomalies détectées (z-score < -1.5).

5. **Croisement avec données environnementales et matériel** :
   - Élimination des faux positifs liés à la météo ou au matériel.
   - 49 sites conservés après filtrage.

6. **Analyse des logs** :
   - TF-IDF sur les rapports de terrain.
   - Identification d'un log atypique sur KNT-KES mentionnant une effraction par des braconniers.

7. **Conclusion** : La baisse de population sur KNT-KES est liée à une activité humaine illégale (braconnage).

## bat_surveys.csv

Dataset concernant des **sites d'observation de chauves-souris** :

| Colonne | Type | Description |
|---------|------|-------------|
| survey_id | str | Identifiant unique du relevé |
| site_code | str | Code du site d'observation |
| survey_start | datetime | Date et heure de début du relevé |
| survey_end | datetime | Date et heure de fin du relevé |
| survey_method | str | Méthode d'observation utilisée |
| observation_hours | float | Nombre d'heures d'observation |
| observed_bat_count | int | Nombre de chauves-souris observées |
| acoustic_detection_count | int | Nombre de détections acoustiques |
| nightly_visit_count | int | Nombre de visites nocturnes |
| colony_estimate | float | Estimation de la taille de la colonie |
| comparison_species_count | int | Nombre d'espèces comparées |
| survey_quality | str | Qualité du relevé |
| weather_interference | float | Degré d'interférence météorologiques |

## 1. Imports et chargement des données


In [22]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import re

In [23]:
bats = pd.read_csv('./data/ecology/bat_surveys.csv')
report = pd.read_csv('./data/ecology/environmental_field_reports.csv')
env = pd.read_csv('./data/ecology/environmental_monthly.csv')
devices = pd.read_csv('./data/ecology/monitoring_device_operations.csv')
sites = pd.read_csv('./data/ecology/monitoring_sites.csv')
orchard = pd.read_csv('./data/ecology/orchard_surveys.csv')

## 2. Nettoyage et préparation des données

Les données qui intéressent le plus l'équipe sont les informations recueillies sur les chauves-souris (bat_surveys.csv).

In [24]:
bats.info()

<class 'pandas.DataFrame'>
RangeIndex: 4994 entries, 0 to 4993
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   survey_id                 4994 non-null   str    
 1   site_code                 4994 non-null   str    
 2   survey_start              4994 non-null   str    
 3   survey_end                4994 non-null   str    
 4   survey_method             4994 non-null   str    
 5   observation_hours         4994 non-null   str    
 6   observed_bat_count        4994 non-null   str    
 7   acoustic_detection_count  4994 non-null   str    
 8   nightly_visit_count       4994 non-null   str    
 9   colony_estimate           4994 non-null   float64
 10  comparison_species_count  4994 non-null   int64  
 11  observer_team             4974 non-null   str    
 12  survey_quality            4994 non-null   str    
 13  weather_interference      4994 non-null   str    
dtypes: float64(1), int6

On peut voir que plusieurs valeurs numériques ont un type str plutôt que int ou float.

In [25]:
print(bats.head(20)['observed_bat_count'][19])

9 count


Et que ces valeurs ne sont pas au même format.

On procède alors au nettoyage des colonnes numériques et on retire tout caractère autre que ceux de 0 à 9, . ou , puis on change leur type en int ou float en fonction du contexte.

In [26]:
float_col = [
    'colony_estimate',
    'weather_interference',
    'observation_hours'
]

int_col = [
    'acoustic_detection_count',
    'nightly_visit_count',
    'observed_bat_count',
    'comparison_species_count'
]

numeric_col = float_col + int_col

def remove_alphabetic_char(data, col_names):
    data = data.copy()
    for name in col_names:
        data[name] = data[name].astype(str).str.replace(r'[^\d.,-]', '', regex=True)
    return data

bats = remove_alphabetic_char(bats, numeric_col)

print(bats['observed_bat_count'].head(20)[19])

9


In [27]:
def col_type_converter(data, col_names, dtype):
    data = data.copy()
    for name in col_names:
        col = data[name]
        if dtype in (float, int):
            col = col.astype(str).str.replace(',', '.', regex=False)
        data[name] = col.astype(dtype)
    return data

bats = col_type_converter(bats, float_col, float)
bats = col_type_converter(bats, int_col, int)

bats.info()

<class 'pandas.DataFrame'>
RangeIndex: 4994 entries, 0 to 4993
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   survey_id                 4994 non-null   str    
 1   site_code                 4994 non-null   str    
 2   survey_start              4994 non-null   str    
 3   survey_end                4994 non-null   str    
 4   survey_method             4994 non-null   str    
 5   observation_hours         4994 non-null   float64
 6   observed_bat_count        4994 non-null   int64  
 7   acoustic_detection_count  4994 non-null   int64  
 8   nightly_visit_count       4994 non-null   int64  
 9   colony_estimate           4994 non-null   float64
 10  comparison_species_count  4994 non-null   int64  
 11  observer_team             4974 non-null   str    
 12  survey_quality            4994 non-null   str    
 13  weather_interference      4994 non-null   float64
dtypes: float64(3), int6

In [28]:
date_col = [
    'survey_start',
    'survey_end'
]

def col_to_date(data, col_names):
    for name in col_names:
        data[name] = pd.to_datetime(data[name], format='mixed')
    return data

bats = col_to_date(bats, date_col)

bats.info()
bats['survey_start'].head()

<class 'pandas.DataFrame'>
RangeIndex: 4994 entries, 0 to 4993
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   survey_id                 4994 non-null   str           
 1   site_code                 4994 non-null   str           
 2   survey_start              4994 non-null   datetime64[us]
 3   survey_end                4994 non-null   datetime64[us]
 4   survey_method             4994 non-null   str           
 5   observation_hours         4994 non-null   float64       
 6   observed_bat_count        4994 non-null   int64         
 7   acoustic_detection_count  4994 non-null   int64         
 8   nightly_visit_count       4994 non-null   int64         
 9   colony_estimate           4994 non-null   float64       
 10  comparison_species_count  4994 non-null   int64         
 11  observer_team             4974 non-null   str           
 12  survey_quality            4994 

0   2019-01-05 18:00:00
1   2019-02-05 18:00:00
2   2019-03-05 18:00:00
3   2019-04-05 18:00:00
4   2019-05-05 18:00:00
Name: survey_start, dtype: datetime64[us]

In [29]:
print(f'Nombre de doublons avant suppression: {bats.duplicated().sum()}')
bats = bats.drop_duplicates()
print(f'Nombre de doublons après suppression: {bats.duplicated().sum()}')

Nombre de doublons avant suppression: 2
Nombre de doublons après suppression: 0


## 3. Détection des anomalies temporelles avec STL

Première approche : décomposition STL (Seasonal-Trend-Loess) suivie d'un calcul de z-score sur les résidus.

In [30]:
bats_date = bats.copy()
bats_date = bats_date.sort_values('survey_start').set_index('survey_start')

stl = STL(bats_date['observed_bat_count'], period=12, robust=True)
res = stl.fit()

bats_date['trend'] = res.trend
bats_date['seasonal'] = res.seasonal
bats_date['residual'] = res.resid

In [31]:
window_size = 5

rolling_mean = bats_date['residual'].rolling(window=window_size, center=True).mean()
rolling_std = bats_date['residual'].rolling(window=window_size, center=True).std()

bats_date['z_score_residual'] = (bats_date['residual'] - rolling_mean) / rolling_std

In [32]:
anomalie_temp = bats_date[bats_date['z_score_residual'] < -1.5]

print('Baisses temporelles anormales détectées:')
print(anomalie_temp[['observed_bat_count', 'residual', 'z_score_residual']])
print('')
print('Sites concernés:')
print(anomalie_temp['site_code'].unique())

Baisses temporelles anormales détectées:
                     observed_bat_count   residual  z_score_residual
survey_start                                                        
2019-01-05 18:00:00                   4  -9.229844         -1.623067
2019-01-05 18:00:00                  14  -1.749284         -1.603005
2019-02-05 18:00:00                   3 -15.350392         -1.656261
2019-03-05 18:00:00                   7 -25.666227         -1.727007
2019-03-05 18:00:00                  14 -13.377198         -1.639263
...                                 ...        ...               ...
2025-04-05 18:00:00                   2 -19.302658         -1.526960
2025-04-05 18:00:00                   6 -13.117825         -1.618162
2025-05-05 18:00:00                   2 -22.319511         -1.587435
2025-05-05 18:00:00                   4 -15.759038         -1.551266
2025-06-05 18:00:00                   9 -11.877330         -1.650890

[155 rows x 3 columns]

Sites concernés:
<StringArray>
['KNT-

Cette approche ne réduit pas très bien le nombre de sites.

In [33]:
numeric_col_env = [
    'mean_temperature_c',
    'rainfall_mm',
    'mean_humidity_pct',
    'vegetation_cover_pct',
    'habitat_disturbance_index',
    'pesticide_index',
    'data_completeness_pct'
]

env = remove_alphabetic_char(env, numeric_col_env)
env = col_type_converter(env, numeric_col_env, float)
env.info()

<class 'pandas.DataFrame'>
RangeIndex: 4994 entries, 0 to 4993
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   site_code                  4994 non-null   str    
 1   reporting_month            4994 non-null   str    
 2   mean_temperature_c         4994 non-null   float64
 3   rainfall_mm                4994 non-null   float64
 4   mean_humidity_pct          4994 non-null   float64
 5   vegetation_cover_pct       4984 non-null   float64
 6   habitat_disturbance_index  4994 non-null   float64
 7   pesticide_index            4979 non-null   float64
 8   land_use_category          4994 non-null   str    
 9   weather_exception_code     204 non-null    str    
 10  data_completeness_pct      4994 non-null   float64
dtypes: float64(7), str(4)
memory usage: 429.3 KB


In [34]:
m = bats_date.merge(env, on=['site_code'], how='left')

features = [
    'observed_bat_count',
    'acoustic_detection_count',
    'nightly_visit_count',
    'mean_temperature_c',
    'rainfall_mm',
    'mean_humidity_pct',
    'vegetation_cover_pct',
    'pesticide_index',
    'habitat_disturbance_index'
]

scaler = StandardScaler()
x_scale = scaler.fit_transform(m[features].fillna(m[features].median()))

iso_forest = IsolationForest(contamination=0.05, random_state=42)
m['anomalie_forest'] = iso_forest.fit_predict(x_scale)

anomalies_multivariees = m[m['anomalie_forest'] == -1]
print('Sites avec anomalies multivariées (IsolationForest):')
print(anomalies_multivariees['site_code'].unique())

Sites avec anomalies multivariées (IsolationForest):
<StringArray>
['AVL-SEY', 'AVL-NMQ', 'AVL-FNQ', 'AVL-MAR', 'TMB-BRC', 'TMB-NAC', 'KNT-GGN',
 'KNT-UJJ', 'KNT-YRX', 'KNT-BME', 'KNT-CYP', 'AVL-GRB', 'AVL-DER', 'AVL-RNY',
 'AVL-RBC', 'KNT-TLQ', 'AVL-YAU', 'KNT-JAK', 'AVL-BWK', 'KNT-XAC', 'KNT-CHP',
 'KNT-LUM', 'AVL-TRP', 'TMB-SXP', 'KNT-JQG', 'KNT-DMJ', 'AVL-KAR', 'KNT-GKR',
 'KNT-VLB', 'KNT-WUQ', 'AVL-VEL', 'AVL-ECJ', 'TMB-HVD', 'AVL-ORR', 'KNT-SEN',
 'TMB-DLQ', 'TMB-HQL', 'TMB-CKU', 'KNT-QWU', 'KNT-YRG', 'AVL-NFR', 'KNT-KES',
 'AVL-HUA', 'AVL-ARU', 'TMB-GZX', 'TMB-AYQ', 'AVL-ABG', 'KNT-TEX', 'TMB-WST']
Length: 49, dtype: str


In [35]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
m['anomalie_lof'] = lof.fit_predict(x_scale)

anomalies_multivariees = m[(m['anomalie_lof'] == -1)]
print('Sites avec anomalies multivariées (LOF):')
print(anomalies_multivariees['site_code'].unique())

Sites avec anomalies multivariées (LOF):
<StringArray>
['AVL-BVM', 'AVL-ABG', 'AVL-VEL', 'AVL-YAU', 'KNT-SEN', 'TMB-NAC', 'KNT-YRG',
 'KNT-VXR', 'AVL-DER', 'TMB-DLQ', 'TMB-VZT', 'TMB-HQL', 'AVL-BWK', 'AVL-RNY',
 'TMB-GWP', 'AVL-MAR', 'KNT-XAC', 'KNT-TEX', 'TMB-WQL', 'AVL-HTY', 'TMB-SXP',
 'TMB-XTD', 'KNT-UJJ', 'KNT-LUM', 'AVL-TRP', 'TMB-SKZ', 'KNT-DMJ', 'AVL-FNQ',
 'KNT-QWU', 'AVL-ECJ', 'TMB-WST', 'KNT-WUQ', 'AVL-HUA', 'AVL-NFR', 'AVL-SEY',
 'AVL-KAR', 'TMB-HVD', 'KNT-NNM', 'KNT-VLB', 'TMB-BRC', 'KNT-JAK', 'AVL-ORR',
 'KNT-CHP', 'KNT-GKR', 'AVL-NMQ', 'AVL-ARU', 'TMB-PEL', 'KNT-YRX', 'KNT-CYP',
 'AVL-GRB', 'TMB-HFR', 'TMB-CKU', 'TMB-GZX', 'TMB-WYT', 'AVL-RBC', 'AVL-FNM',
 'TMB-UVF', 'KNT-KES', 'TMB-AYQ', 'KNT-GGN', 'KNT-BME', 'KNT-TLQ', 'KNT-JQG',
 'TMB-QCB']
Length: 64, dtype: str


Comme le resultat n'etait pas concluant on a décidé de reprendre à zéro et de tester une approche différente.

On se concentre sur le z-score pour repérer des baisses anormales, puis on intègre ensuite les problèmes de matériel et les problèmes temporels.
Enfin, on explore les logs liés aux dates des baisses de population qui ne sont liées ni à une défaillance de matériel d'observation, ni à une météo anormale.

In [36]:
def z_score(s):
    m = s.median()
    e = (s - m).abs().median()
    return 0.6745 * (s - m) / e if e else pd.Series(np.nan, index=s.index)

### Méthode finale : z-score sur les variations mensuelles

D'abord, on regarde les baisses mensuelles de chauves-souris avec le z-score comme critère de comparaison.

Pour cela, on calcule le pourcentage de modification entre la moyenne d'un mois et celle du même mois de l'année précédente. Puis, on utilise ce pourcentage pour calculer un z-score. Si ce z-score montre une diminution anormale, alors il y a une anomalie sur le site liée.

In [37]:
bats_month = bats.copy()
bats_month['month'] = bats['survey_start'].dt.to_period('M')

monthly_avg = bats_month.groupby(['site_code', 'month']).agg({
    'observed_bat_count': ['mean', 'std', 'count']
}).reset_index()
monthly_avg.columns = ['site_code', 'month', 'mean_bats', 'std_bats', 'n_obs']

monthly_avg['month_of_year'] = monthly_avg['month'].dt.month
monthly_avg['year'] = monthly_avg['month'].dt.year

monthly_avg = monthly_avg.sort_values(['site_code', 'month']).reset_index(drop=True)

monthly_avg['prev_mean'] = monthly_avg.groupby(['site_code', 'month_of_year'])['mean_bats'].shift(1)

monthly_avg['pct_change'] = (monthly_avg['mean_bats'] - monthly_avg['prev_mean']) / monthly_avg['prev_mean'] * 100

monthly_avg['z_score'] = z_score(monthly_avg['pct_change'])

anomaly = monthly_avg[monthly_avg['z_score'] < -1.5]
print('Anomalies mensuelles détectées:')
print(anomaly[['site_code', 'month']])

Anomalies mensuelles détectées:
     site_code    month
561    AVL-FNQ  2020-04
579    AVL-FNQ  2021-10
600    AVL-FNQ  2023-07
605    AVL-FNQ  2023-12
615    AVL-FNQ  2024-10
...        ...      ...
4286   TMB-PEL  2025-03
4310   TMB-QCB  2020-09
4443   TMB-SKZ  2025-04
4566   TMB-UVF  2022-07
4819   TMB-WST  2024-02

[72 rows x 2 columns]


## 4. Fusion avec les données environnementales et matériel

Ensuite, on va fusionner la DataFrame sur les chauves-souris avec le DataFrame des informations liées au matériel.

In [38]:
devices = col_type_converter(devices, ['uptime_pct', 'acoustic_quality'], float)
devices = col_to_date(devices, ['operation_start', 'operation_end'])

devices_month = devices.copy()
devices_month['month'] = devices['operation_start'].dt.to_period('M')

device_issues = pd.merge(
    anomaly,
    devices_month,
    on=['site_code', 'month'],
    how='left'
)

env = col_to_date(env, ['reporting_month'])
env_monthly = env
env_monthly['month'] = env['reporting_month'].dt.to_period('M')

devices_env_issues = pd.merge(
    device_issues,
    env_monthly,
    on=['site_code', 'month'],
    how='left'
)

devices.describe()

,operation_start,operation_end,uptime_pct,acoustic_quality
count,4994,4994,4994.000000,4994.000000
mean,2022-03-17 12:36:02.595114,2022-04-16 22:45:35.403684,92.526131,78.991270
min,2019-01-01 00:00:00,2019-01-31 23:59:59,23.100000,20.000000
25%,2020-08-01 00:00:00,2020-08-31 23:59:59,90.200000,73.900000
50%,2022-04-01 00:00:00,2022-04-30 23:59:59,93.800000,79.300000
75%,2023-11-01 00:00:00,2023-11-30 23:59:59,97.300000,84.600000
max,2025-06-01 00:00:00,2025-06-30 23:59:59,100.000000,100.000000
std,NaN,NaN,7.924973,8.505031


On va supprimer tous les sites qui ont soit eu un problème de matériel, soit eu des problèmes météorologiques.

Là encore, on utilise un z-score sur les colonnes habitat_disturbance_index, pesticide_index et rainfall_mm. Pour uptime_pct, on choisit de plutôt faire une élimination si le temps est en dessous de 90 %. La plupart des observations sont à 90 %, donc on enlève les problèmes les plus marqués.

In [39]:
for col in ['habitat_disturbance_index', 'pesticide_index', 'rainfall_mm', 'uptime_pct']:
    med = devices_env_issues[col].median()
    eam = (devices_env_issues[col] - med).abs().median()
    devices_env_issues[f'z_{col}'] = 0.6745 * (devices_env_issues[col] - med) / eam

site_anomalie = []
for _, row in devices_env_issues.iterrows():
    site = row['site_code']
    month = row['month']

    has_material_issue = (
        (row['uptime_pct'] < 90)
    )

    has_env_issue = (
        (row['z_habitat_disturbance_index'] > 1.5) or
        (row['z_pesticide_index'] > 1.5) or
        (row['z_rainfall_mm'] > 1.5)
    )

    if not has_material_issue and not has_env_issue:
        pair = {'site': site, 'period': month}
        if pair not in site_anomalie:
            site_anomalie.append(pair)

print('Sites avec anomalies après filtrage:')
print(site_anomalie)
print(f'Nombre de sites : {len(site_anomalie)}')

devices_env_issues.describe()

Sites avec anomalies après filtrage:
[{'site': 'AVL-FNQ', 'period': Period('2021-10', 'M')}, {'site': 'AVL-FNQ', 'period': Period('2023-07', 'M')}, {'site': 'AVL-FNQ', 'period': Period('2024-10', 'M')}, {'site': 'AVL-GRB', 'period': Period('2020-01', 'M')}, {'site': 'AVL-GRB', 'period': Period('2021-08', 'M')}, {'site': 'AVL-GRB', 'period': Period('2024-11', 'M')}, {'site': 'AVL-GRB', 'period': Period('2025-02', 'M')}, {'site': 'AVL-MAR', 'period': Period('2021-06', 'M')}, {'site': 'AVL-NFR', 'period': Period('2022-11', 'M')}, {'site': 'AVL-NFR', 'period': Period('2022-12', 'M')}, {'site': 'AVL-ORR', 'period': Period('2024-08', 'M')}, {'site': 'AVL-RNY', 'period': Period('2022-09', 'M')}, {'site': 'AVL-VEL', 'period': Period('2020-04', 'M')}, {'site': 'AVL-VEL', 'period': Period('2024-02', 'M')}, {'site': 'KNT-DMJ', 'period': Period('2023-02', 'M')}, {'site': 'KNT-JQG', 'period': Period('2021-02', 'M')}, {'site': 'KNT-KES', 'period': Period('2024-05', 'M')}, {'site': 'KNT-LUM', 'period

,mean_bats,std_bats,n_obs,month_of_year,year,prev_mean,pct_change,z_score,operation_start,operation_end,...,rainfall_mm,mean_humidity_pct,vegetation_cover_pct,habitat_disturbance_index,pesticide_index,data_completeness_pct,z_habitat_disturbance_index,z_pesticide_index,z_rainfall_mm,z_uptime_pct
count,72.000000,0.0,72.0,72.000000,72.000000,72.000000,72.000000,72.000000,72,72,...,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,7.200000e+01
mean,1.944444,NaN,1.0,6.430556,2022.527778,12.194444,-84.825137,-1.716437,2022-12-24 08:40:00,2023-01-23 17:19:59,...,184.662500,78.070833,53.077778,43.913889,31.087500,95.791667,0.101405,-0.249057,-0.057857,-4.462601e-01
min,0.000000,NaN,1.0,1.000000,2020.000000,2.000000,-100.000000,-2.023500,2020-01-01 00:00:00,2020-01-31 23:59:59,...,53.400000,62.300000,15.600000,0.000000,0.000000,88.200000,-2.496833,-2.003745,-1.788777,-1.030145e+01
25%,1.000000,NaN,1.0,3.750000,2021.000000,6.000000,-88.544892,-1.791706,2021-09-23 12:00:00,2021-10-24 05:59:59,...,134.825000,73.375000,40.775000,33.250000,16.100000,94.275000,-0.529542,-1.095004,-0.715049,-6.080720e-01
50%,2.000000,NaN,1.0,7.000000,2023.000000,11.000000,-83.333333,-1.686250,2023-04-16 00:00:00,2023-05-16 11:59:59,...,189.050000,78.600000,54.150000,42.200000,35.500000,95.700000,0.000000,0.000000,0.000000,1.453698e-15
75%,3.000000,NaN,1.0,10.000000,2024.000000,17.000000,-77.777778,-1.573833,2024-05-01 00:00:00,2024-05-31 23:59:59,...,235.850000,82.975000,64.825000,55.600000,43.725000,97.325000,0.792833,0.464248,0.617138,6.642803e-01
max,6.000000,NaN,1.0,12.000000,2025.000000,36.000000,-75.000000,-1.517625,2025-05-01 00:00:00,2025-05-31 23:59:59,...,332.400000,89.500000,98.000000,84.400000,62.700000,100.000000,2.496833,1.535264,1.890314,1.614712e+00
std,1.471694,NaN,0.0,3.491746,1.678130,7.330505,7.946993,0.160807,NaN,NaN,...,69.333861,6.282850,17.201197,20.210605,18.020403,2.433380,1.195794,1.017135,0.914285,2.141763e+00


## 5. Analyse des logs

Ensuite, on trie les logs en fonction des sites qu'on a détectés avec des anomalies, et on filtre aussi par rapport à la période des anomalies, en gardant un mois d'écart car les logs peuvent donner des informations sur un problème qui serait la cause de la baisse ou un constat d'une baisse.

In [40]:
report = col_to_date(report, ['reporting_period'])

report_site = report.copy()

anom = pd.DataFrame(site_anomalie).drop_duplicates()

merged = report.merge(anom, left_on='site_code', right_on='site', how='inner')

merged['ecart_mois'] = (
    (merged['reporting_period'].dt.year - merged['period'].dt.year) * 12
    + (merged['reporting_period'].dt.month - merged['period'].dt.month)
).abs()

report_site = (merged[merged['ecart_mois'] <= 1]
               .drop_duplicates(subset=['site_code', 'reporting_period'])
               .reset_index(drop=True))

print(f'Nombre de logs pertinents: {report_site.shape[0]}')
print(report_site[['site_code', 'reporting_period', 'period', 'ecart_mois']]
      .sort_values(['site_code', 'reporting_period']))

Nombre de logs pertinents: 4
  site_code reporting_period   period  ecart_mois
0   KNT-KES       2024-04-01  2024-05           1
1   KNT-VLB       2020-07-01  2020-07           0
3   KNT-XAC       2023-12-01  2023-11           1
2   KNT-YRG       2024-03-01  2024-04           1


Comme il ne reste que 4 logs, c'est assez simple d'aller vérifier manuellement. Cependant, on va quand même utiliser l'algorithme TF-IDF pour regarder si un ou plusieurs ne sont pas similaires aux autres.

In [41]:
def clean_text(txt):
    txt = txt.lower()  # minuscules
    txt = re.sub(r'[^\w\s]', ' ', txt)  # ponctuation
    txt = re.sub(r'\s+', ' ', txt).strip()
    return txt

report_site['clean'] = report['text'].apply(clean_text)

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
M = tfidf.fit_transform(report_site['clean'])

for i, row in report_site.iterrows():
    scores = pd.Series(M[i].toarray()[0], index=tfidf.get_feature_names_out())
    top = scores.nlargest(8)
    print(f"{row['site_code']} {row['reporting_period']} : {', '.join(top.index)}")

KNT-KES 2024-04-01 00:00:00 : the, was, and, access, not, with, acoustic, and the
KNT-VLB 2020-07-01 00:00:00 : and, the, cycle, maintenance cycle, sensor, sensor maintenance, interval, maintenance
KNT-YRG 2024-03-01 00:00:00 : and, the, habitat, assessment, disturbance assessment, disturbance, habitat disturbance, interval
KNT-XAC 2023-12-01 00:00:00 : and, the, exception, exception review, rainfall, rainfall exception, review, interval


In [42]:
S = cosine_similarity(M)

np.fill_diagonal(S, 0)

report_site['sim_moyenne'] = S.mean(axis=0)
report_site['plus_proche'] = S.max(axis=0)

# Affichage des résultats
print(report_site[['site_code', 'reporting_period', 'sim_moyenne', 'plus_proche']].sort_values('sim_moyenne'))

  site_code reporting_period  sim_moyenne  plus_proche
0   KNT-KES       2024-04-01     0.179422     0.258323
1   KNT-VLB       2020-07-01     0.442582     0.769771
3   KNT-XAC       2023-12-01     0.445734     0.813573
2   KNT-YRG       2024-03-01     0.453895     0.813573


On obtient donc un log sur KNT-KES qui est très différent des autres. Quand on va voir ce qu'il contient, on découvre qu'après une inspection, on a constaté une baisse du nombre de chauves-souris, alors que les instruments de mesure et les conditions météorologiques n'ont rien d'anormal.

On voit aussi qu'il y a eu des traces non identifiées sur le site, ce qui laisse croire à une effraction par des braconniers.

## Conclusion

L'analyse des données ecology a permis de :

1. Détecter des baisses anormales de population de chauves-souris sur plusieurs sites.
2. Tester différentes méthodes (STL, IsolationForest, LOF) avant d'adopter une approche basée sur le z-score des variations mensuelles.
3. Éliminer les fausses anomalies liées à des problèmes matériels ou météorologiques.
4. Identifier une cause humaine : le site KNT-KES montre une baisse inexpliquée par des facteurs naturels, avec des logs mentionnant une effraction par des braconniers.

Piste à transmettre : Activité illégale de braconnage sur le site KNT-KES, à investiguer avec les autorités compétentes.

### Utilisation de l'IA générative

On a utilisé l'IA générative pour ce projet, pour comprendre l'utilisation et la logique derrière certaines notions, ainsi que pour la correction orthographique.

On tient à préciser qu'aucun code n'a été généré par IA pour ce projet.

Voici un exemple de prompt utilisé :

> explique moi step by step le fonctionnement de l'algorithme TF-IDF en pseudocode